In [1]:
import torch
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, InputExample, losses, util
from torch.utils.data import DataLoader
from datasets import load_dataset
from scipy.stats import spearmanr, pearsonr

/opt/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Chargement du dataset complet (4000 lignes)
ds_all = load_dataset("syang687/FinSTS", split="test+validation")

# Split manuel : 3000 pour l'entraînement, 1000 pour la validation
train_size = 3000
train_data = ds_all.select(range(train_size))
val_data = ds_all.select(range(train_size, len(ds_all)))

# Préparation des exemples d'entraînement
train_examples = [
    InputExample(texts=[s1, s2], label=score / 5.0)
    for s1, s2, score in zip(train_data["sentence1"], train_data["sentence2"], train_data["score"])
]

# DataLoader
train_dataloader = DataLoader(
    train_examples,
    shuffle=True,
    batch_size=16,
    collate_fn=SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2").smart_batching_collate
)

# Optimiseur
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
train_loss = losses.CosineSimilarityLoss(model)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)

# Préparation des données de validation
sentences1 = list(val_data["sentence1"])
sentences2 = list(val_data["sentence2"])
human_scores = val_data["score"]  # scores humains à conserver

# Fonction d’évaluation
def evaluate_model(model_name):
    print(f"\n Évaluation du modèle : {model_name}")
    model = SentenceTransformer(model_name)

    emb1 = model.encode(sentences1, convert_to_tensor=True, show_progress_bar=True)
    emb2 = model.encode(sentences2, convert_to_tensor=True, show_progress_bar=True)

    cosine_scores = util.cos_sim(emb1, emb2).diagonal().cpu().numpy()
    spearman_corr, _ = spearmanr(cosine_scores, human_scores)
    pearson_corr, _ = pearsonr(cosine_scores, human_scores)

    print(f"Spearman correlation : {spearman_corr:.4f}")
    print(f"Pearson correlation  : {pearson_corr:.4f}")
    print("-" * 50)


In [3]:
# Chargement du modèle pré-entraîné
model2 = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Encodage des phrases
emb1 = model2.encode(sentences1, convert_to_tensor=True, show_progress_bar=True)
emb2 = model2.encode(sentences2, convert_to_tensor=True, show_progress_bar=True)

# Similarité cosine
cosine_scores = util.cos_sim(emb1, emb2).diagonal().cpu().numpy()

# Corrélations
spearman_corr, _ = spearmanr(cosine_scores, human_scores)
pearson_corr, _ = pearsonr(cosine_scores, human_scores)

print(f"Spearman correlation : {spearman_corr:.4f}")
print(f"Pearson correlation  : {pearson_corr:.4f}")

Batches:   0%|          | 0/32 [00:00<?, ?it/s]/opt/python/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
Batches: 100%|██████████| 32/32 [00:04<00:00,  7.06it/s]


Spearman correlation : 0.8564
Pearson correlation  : 0.9263


In [4]:
model.train()
for epoch in range(10):
    total_loss = 0
    progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}")
    
    for step, batch in enumerate(progress_bar):
        features, labels = batch

        features = [{k: v.to(model.device) for k, v in f.items()} for f in features]
        labels = labels.to(model.device)

        optimizer.zero_grad()
        loss = train_loss(features, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        progress_bar.set_postfix({"Batch Loss": loss.item()})

    avg_loss = total_loss / len(train_dataloader)
    print(f"Epoch {epoch+1} - Avg Loss: {avg_loss:.4f}")

    model.eval()
    with torch.no_grad():
        emb1 = model.encode(sentences1, convert_to_tensor=True, show_progress_bar=False, device=model.device)
        emb2 = model.encode(sentences2, convert_to_tensor=True, show_progress_bar=False, device=model.device)
        
        # Prédictions du modèle (similarité cosine)
        cosine_scores = util.cos_sim(emb1, emb2).diagonal().cpu().numpy()

        # Corrélations
        spearman_corr, _ = spearmanr(cosine_scores, human_scores)
        pearson_corr, _ = pearsonr(cosine_scores, human_scores)

        # Normalisation des scores humains pour calcul de la loss
        val_labels = torch.tensor(human_scores, dtype=torch.float32).to(model.device) / 5.0
        val_preds = torch.tensor(cosine_scores, dtype=torch.float32).to(model.device)

        # Loss de validation (MSE entre prédiction et score humain)
        val_loss = torch.nn.functional.mse_loss(val_preds, val_labels).item()

    print(f"Epoch {epoch+1} - Spearman: {spearman_corr:.4f} | Pearson: {pearson_corr:.4f} | Validation Loss: {val_loss:.4f}")
    print("-" * 60)

    model.train()  # repasser en mode entraînement pour l'epoch suivante

    if epoch == 4:  # on va prendre le 5eme epoch car on a un overfitting à partir de lui
        model.save("finetuned-minilm-similarity-epoch5", safe_serialization=False)
        print("Modèle sauvegardé à 5 epochs")

Epoch 1: 100%|██████████| 188/188 [01:02<00:00,  3.00it/s, Batch Loss=0.0171] 


Epoch 1 - Avg Loss: 0.0190
Epoch 1 - Spearman: 0.8834 | Pearson: 0.9555 | Validation Loss: 0.0133
------------------------------------------------------------


Epoch 2: 100%|██████████| 188/188 [00:57<00:00,  3.29it/s, Batch Loss=0.0179] 


Epoch 2 - Avg Loss: 0.0144
Epoch 2 - Spearman: 0.8857 | Pearson: 0.9574 | Validation Loss: 0.0127
------------------------------------------------------------


Epoch 3: 100%|██████████| 188/188 [00:52<00:00,  3.59it/s, Batch Loss=0.00381]


...

Epoch 8 - Avg Loss: 0.0066
Epoch 8 - Spearman: 0.8834 | Pearson: 0.9546 | Validation Loss: 0.0136
------------------------------------------------------------


Epoch 9: 100%|██████████| 188/188 [00:49<00:00,  3.76it/s, Batch Loss=0.00616]


Epoch 9 - Avg Loss: 0.0062
Epoch 9 - Spearman: 0.8789 | Pearson: 0.9515 | Validation Loss: 0.0143
------------------------------------------------------------


Epoch 10: 100%|██████████| 188/188 [00:51<00:00,  3.64it/s, Batch Loss=0.00515]


Epoch 10 - Avg Loss: 0.0059
Epoch 10 - Spearman: 0.8797 | Pearson: 0.9499 | Validation Loss: 0.0147
------------------------------------------------------------
